# 리포트 52 — 탭을 1~96 으로 늘려도 소거 깊이가 포화하고, 그 대가가 0-도플러 노치다

> ### 한 일
> **ECA 탭 수를 1~96 으로 스윕하며 소거 깊이를 재고, 같은 소거기가 표적의 느린 도플러를 얼마나 함께 지우는지를 속도 문턱으로 환산했다.**

### 결과
1. G3(풀로드) 격자에서 직접파를 기준신호로 합성해 같은 소거기를 걸면 float64 한계까지 내려간다 — 5G 232.3 dB [^1]. ⛔여기 함께 적었던 «다중경로를 넣으면 56.1 dB [^2] 에서 멈춘다» 는 **실내 통제 기하에서 낸 열이라 내렸다**(`archive/chamber_0903/`) — 인용하지 않는다.
2. 그 합성에서 바닥을 정하는 것은 탭 수가 아니라 환경이다 — 탭 1~96 스윕에서 깊이가 포화한다. 직접파를 송신 파형 전체로 합성하면 같은 격자의 깊이가 5G 1.60 dB [^3] 다.
3. 대가는 0-도플러 노치다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^4] 로 세 파형이 같고, 그 무차원 상수는 M 이 정한다 — M = 16 · 48 · 96 에서 0.613 [^5] · 0.596 [^4] · 0.592 [^6] 다.
4. 프레임 48 [^7]개에서 속도 문턱은 WiFi 0.39 m/s [^8] · LTE 1.10 m/s [^9] · 5G 1.16 m/s [^10] 다 — 그보다 빠른 표적이 무는 노치 손실은 3 dB 아래다. ⚠ 같은 프레임 수에서 $T_{CPI}$ 는 48 ms [^11] · 48 ms [^12] · 24 ms [^13] 로 갈린다.
5. 정적 산란체는 ECA 뒤에서 죽은 파라미터다 — 클러터를 100 [^14]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^15] 다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 소거기 | CPI 1회 최소제곱 사영의 standard ECA — 레퍼런스의 지연 사본이 치는 부분공간에 서베일런스를 통째로 사영한다(`src/passive_process.py:13`) |
| 점유 격자 | G3(풀로드) 한 판이다(`benchmark/verify_eca.py:505`) — 그 격자의 기준신호는 WiFi VHT-LTF [^16] · LTE PRS [^17] · 5G NR-PRS [^18] 라, LTE·5G 는 상시 CRS·SSB 자리에 측위 세션 신호가 선다 |
| 깊이 스윕 | 탭 수 1~96 × (직접파만 / 다중경로 포함) 두 조건. ⛔다중경로 열은 실내 통제 기하에서 레이 트레이싱으로 푼 것이고 실측이 아니다(출처 RT [^19]). 두 조건의 차이가 «바닥을 무엇이 정하는가» 를 가른다 |
| 노치 환산 | 3 dB 손실 지점을 $f_d/\Delta f_d$ 무차원으로 재고, 파형별 $\lambda$ 로 속도 문턱으로 옮긴다 |
| 클러터 대조 | 정적 산란체 세기를 배수로 키우며 SCR 변화폭을 잰다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 51 «수신 → ECA → 거리도플러 → CFAR»](51_chain.ipynb) | 사슬의 네 단계와 파형별 형상 |

---

## 직접파를 얼마나 지울 수 있는가

탭을 늘리면 소거가 깊어지다가 멈춘다. 멈추는 자리를 정하는 것이 탭 수인지 환경인지를 가르려고 G3(풀로드) 격자에서 두 조건을 나란히 돌렸다.

| 파형 | 직접파(=기준신호로 합성) | 직접파(=기준신호) + 다중경로(포화) |
|---|---|---|
| WiFi | 202.7 dB [^20] | 33.0 dB [^21] |
| LTE | 219.9 dB [^22] | 41.1 dB [^23] |
| 5G | 232.3 dB [^1] | 56.1 dB [^2] |

왼쪽은 float64 산술의 한계다. ⛔**오른쪽 열은 실내 통제 기하에서 낸 것이라 내렸다**(`archive/chamber_0903/`) — 그 수는 인용하지 않는다. ⛔여기 함께 적었던 「두 열의 간격이 곧 환경이 정하는 몫이다」도 **그 열이 떠받치던 결론이라 뺀다** — 실외 판에서 다시 내기 전에는 이 장면의 바닥을 말하지 않는다.

두 열 모두 직접파를 기준신호로 합성한 판이다(`benchmark/verify_eca.py:146,147`). 헤드라인 사슬은 직접파를 송신 파형 전체(파일럿+데이터)로 합성하고(`benchmark/run_min_cell.py:164`), 그 판의 **시간영역** 깊이는 WiFi 0.35 dB [^24] · LTE 1.33 dB [^25] · 5G 1.60 dB [^3] 다 — 남은 잔류는 열잡음(var=1)보다 WiFi 31.4 dB [^26] · LTE 61.3 dB [^27] · 5G 44.7 dB [^28] 크다.

⚠ 그 잔류가 RD 맵 어디에 서는지는 파형마다 갈린다. 0-도플러 행의 첨두가 잡음 플로어 위로 서는 것은 LTE 한 파형이고(1.5 dB [^29]), WiFi·5G 는 같은 행에서 -122.3 dB [^30] · -137.7 dB [^31] 다. ⛔ 그 두 수는 크기가 아니라 «이 격자에서는 RD 맵에 서지 않았다» 는 표시로 읽고, 왜 LTE 만 남는지는 이 원장 밖의 물음으로 둔다.

⛔ 비-0도플러 첨두(WiFi -164.8 dB [^32] · LTE -41.0 dB [^33] · 5G -180.2 dB [^34])는 0-도플러 첨두를 상수만큼 평행이동한 값이다 — 원장 9줄 전부에서 두 값의 차가 42.50 dB 이고 줄별 폭은 0.09 dB 다. 잔류가 -23.0 ~ +61.3 dB 로 갈리는 9줄에서 그 차가 같으므로, 이 열이 0-도플러 열에 더해 싣는 것은 «0-도플러 행 세 개를 지웠다» 는 규약이다(`benchmark/verify_eca.py:249`) — 사슬의 운용 마스크는 한 행이다(`benchmark/verify_cfar.py:315`).

![f2_eca_depth](../outputs/figures/report04_f2_eca_depth.png)

**그림 1.** ECA 소거 깊이의 바닥을 정하는 것은 무엇인가?

## 대가 — 0-도플러 노치

ECA 는 지연만 다른 성분을 함께 지운다. 느리게 움직이는 표적은 그 성분과 구분되지 않으므로 같이 깎인다. 3 dB 손실 지점은 $f_d/\Delta f_d$ = 0.596 [^4] 이고 세 파형이 같다 — 속도 문턱은 $\lambda$ 와 $\Delta f_d(=1/T_{CPI})$ 둘이 가른다.

| 파형 | $\lambda$ | $T_{CPI}$ (프레임 48 [^7]개) | 3 dB 속도 문턱 (프레임 48 [^7]개) |
|---|---|---|---|
| WiFi | 0.0575 m [^35] | 48 ms [^11] | 0.39 m/s [^8] |
| LTE | 0.1627 m [^36] | 48 ms [^12] | 1.10 m/s [^9] |
| 5G | 0.0857 m [^37] | 24 ms [^13] | 1.16 m/s [^10] |

⚠ 프레임 수를 48 [^7]개로 고정하면 세 파형의 CPI 가 갈린다 — 프레임 길이가 달라 $T_{CPI}$ 가 48 ms [^11] · 48 ms [^12] · 24 ms [^13] 다. 이 표에서 5G 문턱이 LTE 보다 높은 것은 CPI 가 절반이기 때문이고, $\lambda$ 는 5G(0.0857 m [^37])가 LTE(0.1627 m [^36])보다 짧다.

5G 만 프레임 96개로 잡아 CPI 를 48 ms [^11] 로 맞추면 문턱은 WiFi 0.39 m/s [^8] · 5G 0.58 m/s [^38] · LTE 1.10 m/s [^9] 로 $\lambda$ 순서가 된다. ⛔ 이 표의 세 수는 파형이 정한 물리 문턱이 아니라 프레임 수 규약과 함께 정해진 값이다.

![f3_eca_notch](../outputs/figures/report04_f3_eca_notch.png)

**그림 2.** ECA 가 클러터와 함께 지우는 표적의 속도는 얼마인가?

## 정적 클러터는 ECA 뒤에서 죽은 파라미터다

클러터 세기를 100 [^14]배까지 키워도 SCR 변화폭은 3.5e-09 dB [^15] 다. 소거기가 직접파와 함께 정적 성분을 통째로 가져가기 때문이다.

그래서 이 사슬에서 남는 위협은 정적 클러터가 아니라 **표적을 거쳐 오는 성분**이고, 느린 표적은 노치가 먼저 지운다 — 그 축의 결과는 [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) 가 든다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외에서 잰 채널로 소거 바닥을 다시 잰다 | 환경이 정하는 바닥이 실측 채널에서 몇 dB 인지 확정된다 | `benchmark/verify_eca.py` → [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |
| 노치 폭을 CPI 와 함께 스윕한다 | 느린 표적이 노치 밖으로 나오는 CPI 가 수치로 정해진다 | `benchmark/verify_eca.py` → [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 38개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_dpi_db` | 232.3 |
| [^2] | `outputs/verify_eca.json` | `S1_depth_vs_taps[0].rows[12].depth_full_db` | 56.07 |
| [^3] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].depth_tx_dpi_db` | 1.603 |
| [^4] | `outputs/verify_eca.json` | `S4_target_loss[1].fd_3db_over_dfd` | 0.5957 |
| [^5] | `outputs/verify_eca.json` | `S4_target_loss[0].fd_3db_over_dfd` | 0.6129 |
| [^6] | `outputs/verify_eca.json` | `S4_target_loss[2].fd_3db_over_dfd` | 0.5919 |
| [^7] | `outputs/verify_eca.json` | `S4_target_loss[4].M` | 48 |
| [^8] | `outputs/verify_eca.json` | `S4_target_loss[4].v_3db_ms` | 0.3906 |
| [^9] | `outputs/verify_eca.json` | `S4_target_loss[7].v_3db_ms` | 1.104 |
| [^10] | `outputs/verify_eca.json` | `S4_target_loss[1].v_3db_ms` | 1.163 |
| [^11] | `outputs/verify_eca.json` | `S4_target_loss[4].T_cpi_ms` | 48 |
| [^12] | `outputs/verify_eca.json` | `S4_target_loss[7].T_cpi_ms` | 48 |
| [^13] | `outputs/verify_eca.json` | `S4_target_loss[1].T_cpi_ms` | 24 |
| [^14] | `outputs/verify_eca.json` | `S5_clutter_dead.sweep[3].scale` | 100 |
| [^15] | `outputs/verify_eca.json` | `S5_clutter_dead.scr_span_db` | 3.539e-09 |
| [^16] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].ref_name` | VHT-LTF |
| [^17] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].ref_name` | PRS |
| [^18] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].ref_name` | NR-PRS |
| [^19] | `outputs/verify_eca.json` | `meta.setups[0].clutter_src` | RT |
| [^20] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_dpi_db` | 202.7 |
| [^21] | `outputs/verify_eca.json` | `S1_depth_vs_taps[1].rows[12].depth_full_db` | 33.01 |
| [^22] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_dpi_db` | 219.9 |
| [^23] | `outputs/verify_eca.json` | `S1_depth_vs_taps[2].rows[12].depth_full_db` | 41.12 |
| [^24] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].depth_tx_dpi_db` | 0.3466 |
| [^25] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].depth_tx_dpi_db` | 1.333 |
| [^26] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].resid_over_noise_db` | 31.45 |
| [^27] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].resid_over_noise_db` | 61.28 |
| [^28] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].resid_over_noise_db` | 44.71 |
| [^29] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].rd_zerodop_peak_over_nfloor_db` | 1.499 |
| [^30] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].rd_zerodop_peak_over_nfloor_db` | -122.3 |
| [^31] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].rd_zerodop_peak_over_nfloor_db` | -137.7 |
| [^32] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[7].rd_offzero_peak_over_nfloor_db` | -164.8 |
| [^33] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[8].rd_offzero_peak_over_nfloor_db` | -41.02 |
| [^34] | `outputs/verify_eca.json` | `S3_pilot_vs_tx[6].rd_offzero_peak_over_nfloor_db` | -180.2 |
| [^35] | `outputs/verify_eca.json` | `S4_target_loss[4].lam_m` | 0.05754 |
| [^36] | `outputs/verify_eca.json` | `S4_target_loss[7].lam_m` | 0.1627 |
| [^37] | `outputs/verify_eca.json` | `S4_target_loss[1].lam_m` | 0.08565 |
| [^38] | `outputs/verify_eca.json` | `S4_target_loss[2].v_3db_ms` | 0.5777 |